In [14]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import HttpSource, HuggingFaceSource, CompositeSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

# PanTSMini

Almost done!

In [15]:
DATASET_NAME = "PanTSMini"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



17:24:49 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:65


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Abdomen Atlas 3',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung'
]

In [16]:
description = """
We present PanTS (The Pancreatic Tumor Segmentation Dataset) recently created by JHU. It is a large-scale, multi-institutional dataset, containing 36,390 three-dimensional CT volumes from 145 medical centers, with expert-validated, voxel-wise annotations of over 993,000 anatomical structures, covering pancreatic tumors, pancreas head, body, and tail, and 24 surrounding anatomical structures such as vascular/skeletal structures and abdominal/thoracic organs.

As the largest and most comprehensive resource of its kind, PanTS offers a new benchmark for developing and evaluating AI models in pancreatic CT analysis.

This repository presents our training, PanTS-tr (n=9,000), and in-distribution testing images, PanTS-te (n=901).
"""

In [17]:
source = CompositeSource(
    file_type="nifti",
    sources=[
        HttpSource(
            file_type="nifti",
            url="http://www.cs.jhu.edu/~zongwei/dataset/PanTSMini_Label.tar.gz"
        ),
        HuggingFaceSource(
            file_type="nifti",
            repo_id="BodyMaps/PanTSMini",
            description=description,
            post_download=["unzip"]
        ),
    
    ]

)

In [18]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [19]:

print(indexed_datasets.source_config(DATASET_NAME))

CompositeSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='composite',
    sources=[
        HttpSource(
            file_type=<FileType.NIFTI: 'nifti'>,
            post_download=['unzip'],
            source='http',
            url='http://www.cs.jhu.edu/~zongwei/dataset/PanTSMini_Label.tar.gz',
            description=''
        ),
        HuggingFaceSource(
            file_type=<FileType.NIFTI: 'nifti'>,
            post_download=['unzip'],
            source='huggingface',
            repo_id='BodyMaps/PanTSMini',
            description='\nWe present PanTS (The Pancreatic Tumor Segmentation Dataset) recently created by JHU. It
is a large-scale, multi-institutional dataset, containing 36,390 three-dimensional CT volumes from 145 medical 
centers, with expert-validated, voxel-wise annotations of over 993,000 anatomical structures, covering pancreatic 
tumors, pancreas head, body, and tail, and 24 surrounding anatomical structures such as vascular/skeletal 
structures and abdominal/thoracic organs.\n\nAs the largest and most comprehensive resource of its kind, PanTS 
offers a new benchmark for developing and evaluating AI models in pancreatic CT analysis.\n\nThis repository 
presents our training, PanTS-tr (n=9,000), and in-distribution testing images, PanTS-te (n=901).\n'
        )
    ]
)

In [20]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
#downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

In [21]:
# import tarfile

# def extract_all_tar_gz(folder_path, extract_to=None, remove_after=False):
#     """
#     Extract all .tar.gz archives in a folder.
    
#     Args:
#         folder_path (str): Path to folder containing .tar.gz files
#         extract_to (str, optional): Directory to extract files to. 
#                                     If None, extracts to same folder as each archive.
#         remove_after (bool): If True, removes the archive after successful extraction.
    
#     Returns:
#         list: List of extracted file paths
#     """
#     folder = Path(folder_path)
#     extracted_files = []
    
#     # Get all .tar.gz files
#     tar_gz_files = list(folder.glob("*.tar.gz"))
    
#     if not tar_gz_files:
#         print(f"No .tar.gz files found in {folder_path}")
#         return extracted_files
    
#     print(f"Found {len(tar_gz_files)} .tar.gz archives")
    
#     for archive_path in tar_gz_files:
#         try:
#             # Determine extraction directory
#             if extract_to:
#                 extract_dir = Path(extract_to)
#             else:
#                 # Extract to same directory as archive, using archive name (without .tar.gz)
#                 extract_dir = folder / archive_path.stem.replace('.tar', '')
            
#             # Create extraction directory if it doesn't exist
#             extract_dir.mkdir(parents=True, exist_ok=True)
            
#             print(f"Extracting: {archive_path.name} -> {extract_dir}")
            
#             # Extract the archive
#             with tarfile.open(archive_path, 'r:gz') as tar:
#                 tar.extractall(path=extract_dir)
#                 extracted_files.extend([str(extract_dir / member.name) for member in tar.getmembers()])
            
#             # Remove archive if requested
#             if remove_after:
#                 archive_path.unlink()
#                 print(f"Removed: {archive_path.name}")
                
#         except Exception as e:
#             print(f"Error extracting {archive_path.name}: {e}")
    
#     return extracted_files

# extract_all_tar_gz(temp_data_path)

In [22]:
# We have two different scan and mask patterns that i need to capture. We are going to do 2 crawls and combine the indexes.

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="PanTSMini_Image{throwaway}/{reference_id}/ct.nii.gz",
    mask_name_pattern="PanTSMini_Label/{reference_id}/{ROINames}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()

index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="PanTSMini_Image{throwaway}/{reference_id}/ct.nii.gz",
    mask_name_pattern="PanTSMini_Label/{reference_id}/segmentations/{ROINames}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()

index_df2 = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

# drop all scans from the second crawl
index_df2 = index_df2[index_df2["file_type"] != "scan"]




13:25:14 [warning  ] Some files did not match any pattern. [imgtools] unmatched_count=277227 total_count=297029 example_paths=['PanTSMini_Label/PanTS_00000001/segmentations/adrenal_gland_left.nii.gz', 'PanTSMini_Label/PanTS_00000001/segmentations/adrenal_gland_right.nii.gz', 'PanTSMini_Label/PanTS_00000001/segmentations/aorta.nii.gz', 'PanTSMini_Label/PanTS_00000001/segmentations/bladder.nii.gz', 'PanTSMini_Label/PanTS_00000001/segmentations/celiac_artery.nii.gz'] patterns_tried=['PanTSMini_Image{throwaway}/{reference_id}/ct.nii.gz', 'PanTSMini_Label/{reference_id}/{ROINames}.nii.gz'] hint='Check that patterns include all path segments between root and filename.' call=parse_niftis._log_unmatched_summary:189
13:25:39 [warning  ] Some files did not match any pattern. [imgtools] unmatched_count=9901 total_count=297029 example_paths=['PanTSMini_Label/PanTS_00000001/combined_labels.nii.gz', 'PanTSMini_Label/PanTS_00000002/combined_labels.nii.gz', 'PanTSMini_Label/PanTS_00000003/combined_lab

In [23]:
index_df = pd.concat([index_df, index_df2], axis=0, ignore_index=True)
print(len(index_df))
index_df.drop(columns=['throwaway'])



297029

,reference_id,ROINames,filepath,file_type,reference_scan
0,PanTS_00009001,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN
1,PanTS_00009002,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN
2,PanTS_00009003,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN
3,PanTS_00009004,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN
4,PanTS_00009005,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN
...,...,...,...,...,...
297024,PanTS_00009901,prostate,PanTSMini_Label/PanTS_00009901/segmentations/p...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...
297025,PanTS_00009901,spleen,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...
297026,PanTS_00009901,stomach,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...
297027,PanTS_00009901,superior_mesenteric_artery,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...


In [24]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "CT"
    else:
        return "SEG"
    

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["BodyPartExamined"] = "ABDOMEN"


index_df

,throwaway,reference_id,ROINames,filepath,file_type,reference_scan,Modality,BodyPartExamined
0,Te_00009001_00009901,PanTS_00009001,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN
1,Te_00009001_00009901,PanTS_00009002,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN
2,Te_00009001_00009901,PanTS_00009003,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN
3,Te_00009001_00009901,PanTS_00009004,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN
4,Te_00009001_00009901,PanTS_00009005,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN
...,...,...,...,...,...,...,...,...
297024,NaN,PanTS_00009901,prostate,PanTSMini_Label/PanTS_00009901/segmentations/p...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN
297025,NaN,PanTS_00009901,spleen,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN
297026,NaN,PanTS_00009901,stomach,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN
297027,NaN,PanTS_00009901,superior_mesenteric_artery,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN


In [25]:
clinical_data = pd.read_excel(temp_data_path / "metadata.xlsx")

index_df = pd.merge(left=index_df, right=clinical_data, left_on="reference_id", right_on="PanTS ID", how="left")

index_df

,throwaway,reference_id,ROINames,filepath,file_type,reference_scan,Modality,BodyPartExamined,PanTS ID,shape,...,age,manufacturer,manufacturer model,study type,site,site detail,site nationality,study year,tumor?,structured report
0,Te_00009001_00009901,PanTS_00009001,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009001,"(492, 374, 79)",...,NaN,NaN,NaN,NaN,12 Sites,University of Minnesota Health; Rechts der Isa...,DE; NL; CA; IL; FR; US; CN,NaN,1,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
1,Te_00009001_00009901,PanTS_00009002,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009002,"(484, 332, 96)",...,NaN,NaN,NaN,NaN,12 Sites,University of Minnesota Health; Rechts der Isa...,DE; NL; CA; IL; FR; US; CN,NaN,0,FINDINGS: _x000D_\nSpleen: _x000D_\nMean HU va...
2,Te_00009001_00009901,PanTS_00009003,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009003,"(512, 352, 258)",...,NaN,SIEMENS,Sensation 16,COLONOGRAPHY,15 Sites,Mayo Clinic Arizona; Brown University Center f...,US,2000,0,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
3,Te_00009001_00009901,PanTS_00009004,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009004,"(69, 378, 204)",...,62.0,NaN,NaN,NaN,1 Site,University of Minnesota Health,US,NaN,0,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
4,Te_00009001_00009901,PanTS_00009005,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009005,"(491, 294, 88)",...,NaN,NaN,NaN,NaN,1 Site,Memorial Sloan Kettering Cancer Center,US,NaN,1,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297024,NaN,PanTS_00009901,prostate,PanTSMini_Label/PanTS_00009901/segmentations/p...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297025,NaN,PanTS_00009901,spleen,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297026,NaN,PanTS_00009901,stomach,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297027,NaN,PanTS_00009901,superior_mesenteric_artery,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN


In [26]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet


index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df["study year"] = index_df["study year"].astype("string")
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

,throwaway,SampleID,ROINames,filepath,file_type,reference_scan,Modality,BodyPartExamined,PanTS ID,shape,...,age,manufacturer,manufacturer model,study type,site,site detail,site nationality,study year,tumor?,structured report
0,Te_00009001_00009901,PanTS_00009001,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009001,"(492, 374, 79)",...,NaN,NaN,NaN,NaN,12 Sites,University of Minnesota Health; Rechts der Isa...,DE; NL; CA; IL; FR; US; CN,<NA>,1,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
1,Te_00009001_00009901,PanTS_00009002,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009002,"(484, 332, 96)",...,NaN,NaN,NaN,NaN,12 Sites,University of Minnesota Health; Rechts der Isa...,DE; NL; CA; IL; FR; US; CN,<NA>,0,FINDINGS: _x000D_\nSpleen: _x000D_\nMean HU va...
2,Te_00009001_00009901,PanTS_00009003,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009003,"(512, 352, 258)",...,NaN,SIEMENS,Sensation 16,COLONOGRAPHY,15 Sites,Mayo Clinic Arizona; Brown University Center f...,US,2000,0,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
3,Te_00009001_00009901,PanTS_00009004,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009004,"(69, 378, 204)",...,62.0,NaN,NaN,NaN,1 Site,University of Minnesota Health,US,<NA>,0,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
4,Te_00009001_00009901,PanTS_00009005,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009005,"(491, 294, 88)",...,NaN,NaN,NaN,NaN,1 Site,Memorial Sloan Kettering Cancer Center,US,<NA>,1,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297024,NaN,PanTS_00009901,prostate,PanTSMini_Label/PanTS_00009901/segmentations/p...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297025,NaN,PanTS_00009901,spleen,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297026,NaN,PanTS_00009901,stomach,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297027,NaN,PanTS_00009901,superior_mesenteric_artery,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN


In [27]:
index_df

,throwaway,SampleID,ROINames,filepath,file_type,reference_scan,Modality,BodyPartExamined,PanTS ID,shape,...,age,manufacturer,manufacturer model,study type,site,site detail,site nationality,study year,tumor?,structured report
0,Te_00009001_00009901,PanTS_00009001,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009001,"(492, 374, 79)",...,NaN,NaN,NaN,NaN,12 Sites,University of Minnesota Health; Rechts der Isa...,DE; NL; CA; IL; FR; US; CN,<NA>,1,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
1,Te_00009001_00009901,PanTS_00009002,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009002,"(484, 332, 96)",...,NaN,NaN,NaN,NaN,12 Sites,University of Minnesota Health; Rechts der Isa...,DE; NL; CA; IL; FR; US; CN,<NA>,0,FINDINGS: _x000D_\nSpleen: _x000D_\nMean HU va...
2,Te_00009001_00009901,PanTS_00009003,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009003,"(512, 352, 258)",...,NaN,SIEMENS,Sensation 16,COLONOGRAPHY,15 Sites,Mayo Clinic Arizona; Brown University Center f...,US,2000,0,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
3,Te_00009001_00009901,PanTS_00009004,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009004,"(69, 378, 204)",...,62.0,NaN,NaN,NaN,1 Site,University of Minnesota Health,US,<NA>,0,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
4,Te_00009001_00009901,PanTS_00009005,NaN,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,scan,NaN,CT,ABDOMEN,PanTS_00009005,"(491, 294, 88)",...,NaN,NaN,NaN,NaN,1 Site,Memorial Sloan Kettering Cancer Center,US,<NA>,1,FINDINGS: _x000D_\nSpleen: _x000D_\nNormal siz...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297024,NaN,PanTS_00009901,prostate,PanTSMini_Label/PanTS_00009901/segmentations/p...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297025,NaN,PanTS_00009901,spleen,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297026,NaN,PanTS_00009901,stomach,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
297027,NaN,PanTS_00009901,superior_mesenteric_artery,PanTSMini_Label/PanTS_00009901/segmentations/s...,mask,PanTSMini_ImageTe_00009001_00009901/PanTS_0000...,SEG,ABDOMEN,PanTS_00009901,"(461, 318, 156)",...,NaN,SIEMENS,Sensation Open,ABDOMEN,1 Site,National Cancer Institute,US,2010,1,NaN
